# Fine-tuning real de pix2tex en Colab

A diferencia de `colab_prueba_humo.ipynb` (300 formulas, 1 epoca, solo para confirmar que el bucle corre), este notebook:

1. Genera un dataset sintetico mas grande con `entrenamiento/generar_dataset_sintetico.py`.
2. Lo empaqueta al formato `.pkl` que espera `pix2tex`, usando el mismo tokenizer del checkpoint pre-entrenado.
3. Corre el fine-tuning de verdad, mas epocas, guardando los checkpoints directo en Google Drive (Colab borra el disco local al desconectarse).

**Corre primero la prueba de humo si no lo hiciste** -- confirma que el entorno funciona antes de invertir tiempo en un dataset grande.

Antes de empezar: **Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> GPU**.

## 0. (Solo si vas a usar VS Code) Abrir un tunel SSH hacia esta VM

Igual que en la prueba de humo: corre esto una sola vez aqui en el navegador si vas a seguir desde VS Code con Remote-SSH. Si te quedas en el navegador de Colab, saltatela.

In [ ]:
!pip install -q colab-ssh --upgrade
from colab_ssh import launch_ssh_cloudflared
from getpass import getpass

clave_temporal = getpass("Clave temporal para la sesion SSH (no la reutilices): ")
launch_ssh_cloudflared(password=clave_temporal)

## 1. Confirmar que hay GPU asignada

In [ ]:
!nvidia-smi

## 2. Montar Google Drive

El disco de Colab se borra al desconectarse. Los checkpoints del fine-tuning (los `.pth`, chicos) se guardan directo en Drive para no perderlos. El dataset generado (miles de imagenes chicas) se queda en el disco local de Colab -- es rapido de regenerar y escribir miles de archivos chicos en Drive es lento.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/motor-ocr-finetuning"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Checkpoints en:", DRIVE_DIR)

## 3. Clonar el repo

In [ ]:
!git clone https://github.com/rimyortega55-collab/motor-OCR.git
%cd motor-OCR

## 4. Instalar dependencias

`pix2tex` para el modelo/entrenamiento, y una distribucion LaTeX (para `pdflatex`) porque `generar_dataset_sintetico.py` renderiza cada formula a imagen antes de rasterizarla con PyMuPDF. La instalacion de LaTeX tarda varios minutos.

In [ ]:
!pip install -q pix2tex wandb python-Levenshtein pymupdf
!apt-get -qq update
!apt-get -qq install -y texlive-latex-extra texlive-fonts-recommended texlive-latex-recommended
!which pdflatex

## 5. Parametros

Son un punto de partida razonable, no valores probados como optimos -- ajustalos segun cuanto tiempo/GPU tengas y lo que veas en las primeras corridas. Si Colab se queda sin memoria de GPU, baja `BATCHSIZE`/`MICRO_BATCHSIZE` primero.

In [ ]:
N_TRAIN = 3000
N_VAL = 300
PROFUNDIDAD_MAX = 3

EPOCHS = 20
BATCHSIZE = 10
MICRO_BATCHSIZE = 5
LR = 1e-4  # mas bajo que el 1e-3 de la prueba de humo: ya partimos de un checkpoint entrenado, no de cero

## 6. Generar el dataset sintetico

Con `N_TRAIN=3000`/`N_VAL=300` esto puede tardar bastante (renderiza cada formula con `pdflatex` una por una, con reintentos). Si quieres una vuelta rapida primero para revisar que todo el flujo funciona, baja `N_TRAIN`/`N_VAL` en la celda anterior antes de correr esta.

In [ ]:
DATASET_DIR = "entrenamiento/dataset_real"
!python entrenamiento/generar_dataset_sintetico.py --n-train {N_TRAIN} --n-val {N_VAL} --profundidad-max {PROFUNDIDAD_MAX} --out {DATASET_DIR}

## 7. Empaquetar el dataset en `.pkl`

Con el mismo tokenizer que ya usa el checkpoint pre-entrenado -- si se generara un tokenizer nuevo, el vocabulario no coincidiria con los pesos pre-entrenados y el fine-tuning arrancaria de embeddings sin sentido.

In [ ]:
import os
import pix2tex

base_pix2tex = os.path.dirname(pix2tex.__file__)
TOKENIZER = os.path.join(base_pix2tex, "model", "dataset", "tokenizer.json")
CHECKPOINT = os.path.join(base_pix2tex, "model", "checkpoints", "weights.pth")
assert os.path.exists(TOKENIZER), f"No se encontro el tokenizer en {TOKENIZER}"
assert os.path.exists(CHECKPOINT), f"No se encontro el checkpoint en {CHECKPOINT}"

!python -m pix2tex.dataset.dataset -i {DATASET_DIR}/train/imagenes -e {DATASET_DIR}/train/formulas.txt -t {TOKENIZER} -o {DATASET_DIR}/train.pkl
!python -m pix2tex.dataset.dataset -i {DATASET_DIR}/val/imagenes -e {DATASET_DIR}/val/formulas.txt -t {TOKENIZER} -o {DATASET_DIR}/val.pkl

## 8. Armar la config de fine-tuning

Parte de `config_validacion.yaml` (misma arquitectura que el checkpoint pre-entrenado) y sobreescribe lo que cambia para una corrida real: el dataset grande, mas epocas, y que los checkpoints se guarden en Drive.

In [ ]:
import yaml

with open("entrenamiento/config_validacion.yaml") as f:
    config = yaml.safe_load(f)

config.update({
    "data": f"{DATASET_DIR}/train.pkl",
    "valdata": f"{DATASET_DIR}/val.pkl",
    "load_chkpt": CHECKPOINT,
    "tokenizer": TOKENIZER,
    "epochs": EPOCHS,
    "batchsize": BATCHSIZE,
    "micro_batchsize": MICRO_BATCHSIZE,
    "lr": LR,
    "debug": True,  # deja wandb desactivado (ver parse_args en pix2tex/utils/utils.py); cambia a False + wandb login si quieres tracking
    "model_path": f"{DRIVE_DIR}/checkpoints_real",
    "output_path": f"{DRIVE_DIR}/outputs_real",
    "name": "pix2tex_real",
    "test_samples": min(8, N_VAL),
})

with open("entrenamiento/config_real.yaml", "w") as f:
    yaml.safe_dump(config, f)

config

## 9. Correr el fine-tuning

Con estos parametros por defecto esto tarda bastante mas que la prueba de humo (horas, no minutos). Mantene la pestana de Colab abierta -- el entrenamiento en si cuenta como actividad, pero si cerras la pestana el runtime se puede desconectar igual.

In [ ]:
!python entrenamiento/entrenar.py --config entrenamiento/config_real.yaml

## 10. Verificar los checkpoints guardados en Drive

In [ ]:
!ls -la {DRIVE_DIR}/checkpoints_real/pix2tex_real/

## 11. Proximo paso: medir, no asumir

Que el entrenamiento haya corrido y guardado checkpoints **no confirma que el modelo mejoro**. Antes de considerar este fine-tuning listo:

- Corre `pruebas/arnes_evaluacion.py` (o el criterio de calidad que uses para LaTeX) comparando el checkpoint nuevo contra los pesos pre-entrenados originales, sobre un conjunto de prueba que el modelo no haya visto en el fine-tuning.
- El dataset sintetico de este notebook no incluye ruido de escaneo/fotografia real ni el estilo tipografico real de tus PDF -- si la mejora no se sostiene sobre formulas reales, hace falta un dataset con ejemplos reales (o mas realistas) antes de dar el fine-tuning por bueno.
- Recorda la prioridad del proyecto: LaTeX primero, hasta que cumpla el criterio de calidad -- recién despues Markdown.